# Local RAG Pipeline

Run the checked-in NIST PDF corpus through the same tokenizer-based ingestion, persistent Chroma/SQLite store, retrieval, extractive generation, and response validation used by the API. This notebook requires no provider credentials and uses a temporary database directory. The saved outputs predate the latest local-extractor refinement; re-execute to refresh the displayed answer.

In [1]:
from pathlib import Path
import sys
from tempfile import TemporaryDirectory

ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "app").is_dir())
sys.path.insert(0, str(ROOT))

from app.config import Settings
from app.generation import Generator
from app.services import RAGService
from app.storage import VectorStore

corpus = sorted((ROOT / "corpus").glob("*.pdf"))
assert corpus, "No checked-in PDF corpus found"
workspace = TemporaryDirectory(prefix="rag-notebook-")
settings = Settings(
    data_dir=Path(workspace.name),
    embedding_provider="local",
    embedding_model="intfloat/e5-small-v2",
    chunk_size=500,
    chunk_overlap=100,
    min_retrieval_score=None,
)
store = VectorStore(
    data_dir=settings.data_dir,
    embedding_model=settings.embedding_model,
    embedding_provider=settings.embedding_provider,
)
generator = Generator(api_key=None, model="local-extractive")
service = RAGService(store, generator, settings)
print(f"Corpus files: {[path.name for path in corpus]}")

Corpus files: ['NIST_AI_RMF_1.0.pdf', 'NIST_Generative_AI_Profile.pdf']


## Ingest and persist

The service validates each PDF, extracts page metadata, chunks by the local E5 tokenizer, creates embeddings, and stores document metadata in SQLite alongside vectors in Chroma.

In [2]:
documents = []
for path in corpus:
    document = service.upload_document(path.name, path.read_bytes())
    documents.append(document)
    print(f"{document.filename}: {document.chunk_count} chunks, {document.size_bytes:,} bytes")

print("Ready document IDs:", store.list_document_ids())

/home/codespace/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12818.54it/s]

NIST_AI_RMF_1.0.pdf: 70 chunks, 1,946,127 bytes


NIST_Generative_AI_Profile.pdf: 289 chunks, 1,174,643 bytes
Ready document IDs: ['217d5346-1bc3-4a5a-b4da-1d41d45e94a3', '37dcd576-fa0f-4a3f-bdba-0e6f6cc4de30']


## Retrieve and answer

Try a question whose answer appears in the NIST corpus. The response includes both cited chunks and all retrieved chunks with their source-page metadata and similarity scores.

In [3]:
question = "What are the four functions of the NIST AI Risk Management Framework?"
response = service.ask(question, top_k=5)
print(response.answer)
print(f"\nGrounded: {response.grounded}; refused: {response.refusal}")
print("\nCitations:")
for citation in response.citations:
    print(f"- {citation.filename}, page {citation.page}, score {citation.score:.3f}")
print(f"\nRetrieved chunks shown: {len(response.retrieved_chunks)}")

Additional resources related to the Framework are included in the AI RMF Playbook, which is available via the NIST AI RMF website: https://www.nist.gov/itl/ai-risk-management-framework. Priority research and additional guidance that will enhance this Framework will be captured in an associated AI Risk Management Framework Roadmap to which NIST and the broader community can contribute. NIST AI 100-1 Artificial Intelligence Risk Management Framework (AI RMF 1.0) While GOVERN applies to all stages of organizations’ AI risk management pro- cesses and procedures, the MAP, MEASURE, and MANAGE functions can be applied in AI system-specific contexts and at specific stages of the AI lifecycle. [1] [2]

Grounded: True; refused: False

Citations:
- NIST_AI_RMF_1.0.pdf, page 1, score 0.897
- NIST_AI_RMF_1.0.pdf, page 8, score 0.906

Retrieved chunks shown: 5


## Verify cleanup

Delete the indexed PDFs and confirm the registry is empty. The notebook's temporary persistence directory is removed when the Python session ends or `workspace.cleanup()` is called.

In [4]:
for document in documents:
    service.delete_document(document.id)

assert store.list_document_ids() == []
workspace.cleanup()
print("All indexed documents deleted; temporary workspace cleaned up.")

All indexed documents deleted; temporary workspace cleaned up.
